# Tensor Multiplication

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

This tutorial shows the multiplication-type operations that combine tensors with matrices, vectors, and other tensors: `ttm`, `ttv`, `ttt`, `ttsv`, `mttkrp`, `mttkrps`, `innerprod`, `contract`, and `khatrirao`. Each is shown for the tensor types that support it, along with the shape of the result.

[API reference](../utilities.rst), [tensor](../tensor.rst), [sptensor](../sptensor.rst), [ktensor](../ktensor.rst), [ttensor](../ttensor.rst)

In [ ]:
from __future__ import annotations

import numpy as np

import pyttb as ttb

## Overview
In the examples below, the tensors all have shape $(3, 4, 2)$: a dense tensor, the same data as a sparse tensor, a Kruskal tensor, and a Tucker tensor.

| Operation | `tensor` | `sptensor` | `ktensor` | `ttensor` |
|---|---|---|---|---|
| `ttm` (tensor times matrix) | yes | yes | | yes |
| `ttv` (tensor times vector) | yes | yes | yes | yes |
| `ttt` (tensor times tensor) | yes | | | |
| `ttsv` (tensor times same vector) | yes | | | |
| `mttkrp`, `mttkrps` (matricized tensor times Khatri-Rao product) | yes | `mttkrp` | `mttkrp` | `mttkrp` |
| `innerprod` | yes | yes | yes | yes |
| `contract` | yes | yes | | |

### Setup

In [ ]:
def shape_of(t):
    # Show shapes as plain python integers
    return tuple(int(n) for n in t.shape)


rng = np.random.default_rng(0)
shape = (3, 4, 2)

# A dense tensor, and the same data as a sparse tensor
data = rng.random(shape)
data[rng.random(shape) < 0.5] = 0  # zero out about half of the entries
X_dense = ttb.tensor(data)
X_sparse = X_dense.to_sptensor()

# A Kruskal tensor and a Tucker tensor of the same shape
X_ktensor = ttb.ktensor([rng.random((n, 2)) for n in shape])
X_ttensor = ttb.ttensor(
    ttb.tensor(rng.random((2, 2, 2))), [rng.random((n, 2)) for n in shape]
)
print(X_dense.shape, X_sparse.nnz)

## Tensor times matrix (`ttm`)
The mode-$n$ product $\mathcal{X} \times_n A$ multiplies every mode-$n$ fiber by the matrix $A$. If $A$ has shape $(J, I_n)$ the size of mode $n$ changes from $I_n$ to $J$. Use `transpose=True` to multiply by $A^T$ instead, and pass a list of matrices along with `dims` to multiply in several modes.

In [ ]:
A = rng.random((5, 4))  # maps mode 1 from size 4 to size 5
print(shape_of(X_dense.ttm(A, 1)))
print(shape_of(X_sparse.ttm(A, 1)))
print(shape_of(X_ttensor.ttm(A, 1)))

# transpose=True multiplies by A.T (shape (4, 5)), mapping mode 1 from 5 back to 4
Y_dense = X_dense.ttm(A, 1)
print(shape_of(Y_dense.ttm(A, 1, transpose=True)))

# Several modes at once, here modes 0 and 2
B = [rng.random((6, 3)), rng.random((7, 2))]
print(shape_of(X_dense.ttm(B, [0, 2])))

## Tensor times vector (`ttv`)
Multiplying by a vector in mode $n$ removes that mode. Multiplying in every mode gives a scalar. The `dims` argument selects the modes and `exclude_dims` selects all but the given modes.

In [ ]:
v = rng.random(4)
print(shape_of(X_dense.ttv(v, 1)))
print(shape_of(X_sparse.ttv(v, 1)))
print(shape_of(X_ktensor.ttv(v, 1)))
print(shape_of(X_ttensor.ttv(v, 1)))

# One vector for every mode gives a scalar
vectors = [rng.random(n) for n in shape]
print(X_dense.ttv(vectors))
print(X_sparse.ttv(vectors))
print(X_ktensor.ttv(vectors))
print(X_ttensor.ttv(vectors))

# Skip a mode with exclude_dims: the result is a vector over mode 1
print(shape_of(X_dense.ttv([vectors[0], vectors[2]], exclude_dims=1)))

## Tensor times tensor (`ttt`)
`ttt` contracts two tensors over the dimensions given by `selfdims` and `otherdims`, which must have matching sizes. With no dimensions, it computes the outer product. The result has the remaining dimensions of the first tensor followed by those of the second.

In [ ]:
Z_dense = ttb.tensor(rng.random((4, 2, 5)))
# Contract modes 1 and 2 of X (sizes 4 and 2) with modes 0 and 1 of Z
print(shape_of(X_dense.ttt(Z_dense, np.array([1, 2]), np.array([0, 1]))))

# If only selfdims is given it is used for both tensors
print(X_dense.ttt(X_dense, np.array([0, 1, 2])))

# No dimensions: the outer product
print(shape_of(X_dense.ttt(Z_dense)))

## Tensor times the same vector in multiple modes (`ttsv`)
For a cubical tensor, `ttsv` multiplies the same vector in every mode except the first `skip_dim + 1`. With no `skip_dim` the result is a scalar.

In [ ]:
X_cube = ttb.tensor(rng.random((3, 3, 3)))
x = rng.random(3)
print(X_cube.ttsv(x))  # scalar: x multiplied in all modes
print(shape_of(X_cube.ttsv(x, 0)))  # vector: x multiplied in all but the first mode
print(shape_of(X_cube.ttsv(x, 1)))  # matrix

## Matricized tensor times Khatri-Rao product (`mttkrp`, `mttkrps`)
`mttkrp(U, n)` computes the product of the mode-$n$ matricization of the tensor with the Khatri-Rao product of all factor matrices in `U` except the $n$-th. It is the bottleneck of CP-ALS. The result has $I_n$ rows and one column for each column of the factor matrices. For a dense tensor, `mttkrps` computes the result for every mode at once.

In [ ]:
U = [rng.random((n, 2)) for n in shape]
print(shape_of(X_dense.mttkrp(U, 1)))
print(shape_of(X_sparse.mttkrp(U, 1)))
print(shape_of(X_ktensor.mttkrp(U, 1)))
print(shape_of(X_ttensor.mttkrp(U, 1)))

# All modes at once; the results agree with the individual calls
results = X_dense.mttkrps(U)
print([r.shape for r in results])
print(all(np.allclose(results[n], X_dense.mttkrp(U, n)) for n in range(3)))

## Inner product (`innerprod`)
The inner product of two tensors with the same shape is the sum of the products of their entries. It can be computed between any pair of tensor types, exploiting the structure of each.

In [ ]:
# The sparse and dense versions of the same data have the same inner product
print(X_dense.innerprod(X_dense), X_sparse.innerprod(X_dense))

# Compare against the dense computation for the structured tensors
dense_k = X_ktensor.full()
print(np.isclose(X_dense.innerprod(X_ktensor), X_dense.innerprod(dense_k)))
print(np.isclose(X_ktensor.innerprod(X_ktensor), dense_k.innerprod(dense_k)))
dense_t = X_ttensor.full()
print(np.isclose(X_ttensor.innerprod(X_sparse), dense_t.innerprod(X_dense)))

## Contraction (`contract`)
`contract(i, j)` sums over the diagonal of two modes of equal size (a generalized trace). The two modes are removed from the result.

In [ ]:
X_square = ttb.tensor(rng.random((3, 3, 2)))
print(shape_of(X_square.contract(0, 1)))
print(shape_of(X_square.to_sptensor().contract(0, 1)))

## Khatri-Rao product (`khatrirao`)
The Khatri-Rao product of matrices with the same number of columns is the column-wise Kronecker product. This is the matrix that `mttkrp` multiplies implicitly, and it is useful for forming the matricized form of a Kruskal tensor.

In [ ]:
A1, A2, A3 = U
KR = ttb.khatrirao(A1, A2, A3)
print(shape_of(KR))  # (3 * 4 * 2, 2)

# The reverse option reverses the order of the matrices
print(np.allclose(ttb.khatrirao(A3, A2, A1, reverse=True), KR))

# Matricizing a ktensor: mode-0 unfolding equals A1 @ khatrirao(A2, A3, reverse=True).T
K = ttb.ktensor([A1, A2, A3])
unfolded = K.full().to_tenmat(np.array([0])).double()
print(np.allclose(unfolded, A1 @ ttb.khatrirao(A3, A2).T))

## Summary
* `ttm`, `ttv`, `ttt`, `ttsv`, `mttkrp`, `mttkrps`, `innerprod`, and `contract` are methods of the tensor classes; see the [tensor](../tensor.rst), [sptensor](../sptensor.rst), [ktensor](../ktensor.rst), and [ttensor](../ttensor.rst) API pages.
* `khatrirao` is a function; see the [utilities](../utilities.rst) API page.